# SkillMatch: modelo predictivo en Google Colab

Este cuaderno corre la parte de **minería de datos** de la propuesta: arma la tabla de ejemplos, entrena el
árbol de decisión, lo examina con 5 pliegues contra la regresión logística y el «modelo flojo», y genera
`modelo-predictivo.json`. Al final puedes ver la página «Cómo aprende SkillMatch» aquí mismo.

**Cómo usarlo:** menú **Entorno de ejecución → Ejecutar todas**. La celda 1 descarga el proyecto
SkillMatch desde GitHub (no hay que subir nada).

## 1. Descargar el proyecto

In [ ]:
import os, subprocess

# El proyecto completo de SkillMatch; el modelo esta en mineria-datos/modelo-predictivo
if not os.path.isdir('/content/Skillmatch'):
    !git clone -q --depth 1 https://github.com/jesuscastilla-cloud/Skillmatch.git /content/Skillmatch
os.chdir('/content/Skillmatch/mineria-datos/modelo-predictivo')
print('Carpeta lista:', os.getcwd())
print(sorted(os.listdir('.')))


## 2. Instalar lo que falta
Colab ya trae pandas, numpy, scikit-learn y matplotlib. Solo falta `pymongo` (para la base de Render).

In [ ]:
!pip install -q -r 2-modelo/requirements.txt

## 3. Paso 1: la tabla de ejemplos

**Opción A (por defecto): los datos por defecto del `DataSeeder.java`.** No necesita la base.

**Opción B: los datos de la plataforma (Render).** Cambia `USAR_BASE = True`. Te pedirá la `MONGODB_URI`
en un campo oculto: no la escribas en el código ni la guardes en el cuaderno.

In [ ]:
USAR_BASE = False   # True para leer la base de Render

import getpass, subprocess
if USAR_BASE:
    os.environ['MONGODB_URI'] = getpass.getpass('MONGODB_URI (no se muestra): ')
    !cd 1-datos && python exportar_desde_mongo.py
    DATOS = '../1-datos/datos/ejemplos.csv'
    os.environ.pop('MONGODB_URI', None)
else:
    !cd 1-datos && python generar_datos_semilla.py
    DATOS = '../1-datos/datos/ejemplos_semilla.csv'
print('Tabla:', DATOS)

### Mirar la tabla

In [ ]:
import pandas as pd
tabla = pd.read_csv(DATOS.replace('../', ''))
print(tabla.shape[0], 'postulaciones respondidas;', int(tabla.aceptada.sum()), 'aceptadas')
tabla.head(10)

In [ ]:
tabla.describe().T.round(2)

## 4. Pasos 2 y 3: aprender, examinarse y guardar

Árbol CART (4 niveles, 5 casos por hoja), regresión logística y modelo flojo, con validación cruzada de 5
pliegues **por vacante**: todas las postulaciones de una vacante quedan en el mismo grupo, para que el modelo
nunca se examine con una vacante que ya estudió. Guarda `SkillMatch/src/assets/data/modelo-predictivo.json`
(lo usa la página si el servidor no responde) y `2-modelo/resultados/`.

In [ ]:
extra = ''   # pon '--sin-ejemplos' para dejar fuera las vacantes "(ejemplo)"
!cd 2-modelo && python entrenar_modelo.py --datos {DATOS} {extra}

## 5. Resultados para el informe

In [ ]:
pd.read_csv('2-modelo/resultados/metricas.csv', index_col=0)

In [ ]:
print(open('2-modelo/resultados/arbol.txt', encoding='utf-8').read())

In [ ]:
from IPython.display import Image, display
for f in ['fig_confusion.png', 'fig_arbol.png']:
    ruta = '2-modelo/resultados/' + f
    if os.path.exists(ruta):
        display(Image(ruta))

In [ ]:
import json
m = json.load(open('../../SkillMatch/src/assets/data/modelo-predictivo.json', encoding='utf-8'))
print('Estado:', m['estado'])
if m['estado'] == 'listo':
    e = m['examen']
    print('Ganador por F1:', e['ganador'])
    print('¿El árbol le gana al modelo flojo?', 'sí' if e['le_gana_al_flojo'] else 'NO')
    print('Qué pesa más:')
    for v in m['importancia']:
        print(f"  {v['nombre']:<40} {v['valor']*100:5.1f} %")
else:
    print('Faltan', m['datos']['faltan'], 'respuestas para el mínimo.')

## 6. Comprobar que el navegador da lo mismo que Python
Colab ya trae Node.

In [ ]:
!node pruebas/comparar-python-y-javascript.js

## 7. Ver la página «Cómo aprende SkillMatch» aquí mismo (opcional)

Abre en esta celda el frontend del proyecto que se descargó en el paso 1, con el modelo que acabas de
entrenar. Sin el servidor, la página usa ese `modelo-predictivo.json`.

In [ ]:
subprocess.Popen(['python', '-m', 'http.server', '8000', '--directory', '/content/Skillmatch/SkillMatch/src'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

from google.colab import output
output.serve_kernel_port_as_iframe(8000, path='/pages/como-aprende.html', height=900)


Si el cuadro sale pequeño, esta celda te da un enlace para abrir la página en otra pestaña del navegador.
(Funciona mientras el cuaderno esté abierto y conectado.)

In [ ]:
# Enlace para abrir la página en otra pestaña (la forma que Colab recomienda hoy)
from google.colab.output import eval_js
url = eval_js('google.colab.kernel.proxyPort(8000)')
print('Abre esta dirección:', url.rstrip('/') + '/pages/como-aprende.html')

## 8. Descargar lo generado

Descarga el JSON nuevo y los resultados. El JSON va en `SkillMatch/src/assets/data/` del proyecto.

In [ ]:
shutil.make_archive('/content/resultados-modelo', 'zip', '.', '2-modelo/resultados')
from google.colab import files
files.download('../../SkillMatch/src/assets/data/modelo-predictivo.json')
files.download('/content/resultados-modelo.zip')

# 9. Ejercicios para aprender y comprobar que el modelo está bien hecho

Estos ejercicios **no cambian** la página ni el `modelo-predictivo.json`: trabajan con copias en memoria.
Necesitan que hayas corrido las secciones 1 a 4.

La idea de fondo: hoy el modelo **no le gana al modelo flojo**. ¿Es porque el modelo está mal o porque los
datos no tienen nada que enseñar? Para saberlo, los científicos de datos hacen **dos controles**:

- **Control positivo:** le damos datos donde *sabemos* que hay un patrón. Si el modelo está bien, debe encontrarlo.
- **Control negativo:** le damos datos donde *sabemos* que no hay patrón. Debe sacar lo mismo que adivinar.

In [ ]:
# Herramientas comunes para todos los ejercicios (las mismas reglas del PDF)
import numpy as np, pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate
import sys
sys.path.insert(0, '2-modelo')
from entrenar_modelo import pliegues_por_vacante   # el mismo examen por vacante del servidor

base = pd.read_csv('1-datos/datos/ejemplos_semilla.csv')
COLS = [c for c in base.columns if c not in ('vacante', 'es_ejemplo', 'aceptada')]

def cv_de(y, grupos=None):
    """Los 5 grupos del examen, por vacante (grupos = la columna 'vacante')."""
    grupos = base.vacante if grupos is None else grupos
    p = pliegues_por_vacante(y, grupos)
    return [(np.where(p != g)[0], np.where(p == g)[0]) for g in range(5) if (p == g).any()]

def arbol(prof=4, hoja=5):
    return DecisionTreeClassifier(max_depth=prof, min_samples_leaf=hoja, random_state=42)

def examen(X, y, modelos=None, grupos=None):
    modelos = modelos or {'árbol': arbol(),
                          'logística': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
                          'flojo': DummyClassifier(strategy='most_frequent')}
    filas = {}
    for n, m in modelos.items():
        r = cross_validate(m, X, y, cv=cv_de(y, grupos), scoring=['accuracy', 'f1', 'roc_auc'])
        filas[n] = {'exactitud': r['test_accuracy'].mean(), 'F1': r['test_f1'].mean(), 'ROC-AUC': r['test_roc_auc'].mean()}
    return pd.DataFrame(filas).T.round(3)

print('Listo. Columnas que mira el modelo:', len(COLS))

## Ejercicio 1. Control positivo: datos con un patrón conocido

Tomamos las mismas 151 postulaciones, pero **inventamos** la respuesta con una regla que nosotros escogemos:
*quien tiene al menos una de las habilidades pedidas casi siempre es aceptado, y quien no tiene ninguna casi siempre es rechazado* (con un poco de azar,
porque en la vida real nada es perfecto). **Estos datos son de práctica: nunca van a la página.**

**Qué deberías ver:**
- el árbol le gana al flojo por mucho (exactitud cerca de 0,8 o más; ROC-AUC alto);
- la **primera pregunta** del árbol es sobre las habilidades: `pct_habilidades` o `nivel_dominio`. Aquí las dos
  dicen lo mismo, porque quien no tiene ninguna habilidad pedida tiene nivel 0. El árbol escoge cualquiera de las dos;
- en «qué pesa más», esa columna queda muy arriba (más de 0,8);
- la regresión logística también gana, y puede que saque mejor F1 que el árbol. En ese caso SkillMatch usaría
  la logística, como dice el PDF.

Si pasa eso, el modelo **sí sabe encontrar patrones**: el problema de hoy son los datos, no el código.

In [ ]:
rng = np.random.default_rng(7)
positivo = base.copy()
p_aceptada = np.where(positivo.pct_habilidades > 0, 0.85, 0.15)     # la regla que escogimos
positivo['aceptada'] = (rng.random(len(positivo)) < p_aceptada).astype(int)
print('Aceptadas:', positivo.aceptada.sum(), 'de', len(positivo))

display(examen(positivo[COLS], positivo.aceptada))
m = arbol().fit(positivo[COLS], positivo.aceptada)
print(export_text(m, feature_names=COLS, decimals=1, max_depth=2))
print(pd.Series(m.feature_importances_, index=COLS).sort_values(ascending=False).head(3).round(2))

## Ejercicio 2. Control negativo: barajar las respuestas

Ahora hacemos lo contrario: **revolvemos** la columna `aceptada` al azar. Así rompemos cualquier relación
entre los datos y la respuesta.

**Qué deberías ver:** el árbol queda alrededor de 0,5 de exactitud y el ROC-AUC alrededor de 0,5, igual que el
flojo. Compáralo con el resultado real de la sección 4: **es el mismo comportamiento**. Esa es la prueba de que
los datos por defecto se comportan como datos sin patrón.

In [ ]:
negativo = base.copy()
negativo['aceptada'] = np.random.default_rng(1).permutation(negativo.aceptada.values)
display(examen(negativo[COLS], negativo.aceptada))

## Ejercicio 3. ¿Fue mala suerte? Repetir con otras semillas

Los datos por defecto se generan al azar. Si cambiamos la semilla salen otras 300 postulaciones. Corremos
el examen 10 veces.

**Qué deberías ver:** la exactitud del árbol sube y baja alrededor de 0,5 (a veces 0,45, a veces 0,55) y casi
nunca le gana al flojo por más de 2 puntos. No fue mala suerte: es así siempre con estos datos.

In [ ]:
import subprocess
notas = []
for semilla in range(1, 11):
    salida = f'/tmp/semilla_{semilla}.csv'
    subprocess.run(['python', 'generar_datos_semilla.py', '--semilla', str(semilla), '--salida', salida],
                   cwd='1-datos', check=True, capture_output=True)
    d = pd.read_csv(salida)
    t = examen(d[COLS], d.aceptada, grupos=d.vacante)
    notas.append({'semilla': semilla, 'respondidas': len(d),
                  'árbol': t.loc['árbol', 'exactitud'], 'flojo': t.loc['flojo', 'exactitud'],
                  'ROC-AUC árbol': t.loc['árbol', 'ROC-AUC']})
notas = pd.DataFrame(notas)
display(notas)
print('Promedio árbol:', notas['árbol'].mean().round(3), '| promedio flojo:', notas['flojo'].mean().round(3))

## Ejercicio 4. ¿Por qué el PDF limita el árbol a 4 niveles? (sobreajuste)

Con los datos del ejercicio 1 probamos árboles de 1 a 10 niveles y medimos dos notas:
**en los datos con que estudió** y **en el examen** (datos que no vio).

**Qué deberías ver:** con 1 o 2 niveles ya encuentra la regla. Con más niveles la nota de estudio sigue
subiendo hacia 1,0 (se **aprende de memoria** el azar que le pusimos), pero la nota del examen se queda quieta
o baja. A eso se le llama **sobreajuste**. Los límites del PDF
(4 niveles, 5 casos por hoja) existen para evitarlo.

In [ ]:
filas = []
for prof in range(1, 11):
    r = cross_validate(arbol(prof, hoja=1), positivo[COLS], positivo.aceptada, cv=cv_de(positivo.aceptada),
                       scoring='accuracy', return_train_score=True)
    filas.append({'niveles': prof, 'nota estudiando': r['train_score'].mean(), 'nota en el examen': r['test_score'].mean()})
sobre = pd.DataFrame(filas).set_index('niveles').round(3)
display(sobre)
sobre.plot(marker='o', title='Más niveles: memoriza más, pero no acierta más', ylim=(0.4, 1.02));

## Ejercicio 5. ¿Por qué no basta con la exactitud? (datos desbalanceados)

Imagina que las empresas rechazan casi todo: 9 de cada 10. El modelo flojo dice siempre «rechazada» y saca
**0,9 de exactitud** sin aprender nada. Por eso el PDF elige el modelo con **F1**, que mira si encuentra
las aceptadas.

**Qué deberías ver:** el flojo con exactitud alta (cerca de 0,9) pero **F1 = 0**.

In [ ]:
desbalance = base.copy()
desbalance['aceptada'] = (np.random.default_rng(3).random(len(base)) < 0.1).astype(int)
print('Aceptadas:', desbalance.aceptada.sum(), 'de', len(desbalance))
display(examen(desbalance[COLS], desbalance.aceptada))

## Ejercicio 6. Un error que hay que evitar: la fuga de datos

¿Qué pasa si, por error, el modelo puede ver algo que solo se sabe *después* de la respuesta? Aquí le
damos una columna copiada de la respuesta.

**Qué deberías ver:** exactitud 1,0, «perfecto». Es una trampa: en la vida real ese dato no existe al
postularse. Por eso `caracteristicas.py` solo usa lo que se sabe **en el momento de postularse**, y por eso
`es_ejemplo` nunca entra al modelo.

In [ ]:
fuga = base.copy()
fuga['dato_tramposo'] = fuga.aceptada
display(examen(fuga[COLS + ['dato_tramposo']], fuga.aceptada))

## Ejercicio 7. Seguir el árbol a mano

Toma una postulación de la tabla real y recorre `2-modelo/resultados/arbol.txt` con lápiz: en cada
pregunta, si el valor es **menor o igual** al corte vas arriba (`<=`), si es mayor vas abajo (`>`).

**Qué deberías ver:** la hoja a la que llegas tiene la misma probabilidad que la columna `prob_arbol` de
`predicciones.csv`. Es exactamente lo que hace `modelo-predictivo.js` en el navegador.

In [ ]:
pred = pd.read_csv('2-modelo/resultados/predicciones.csv')
fila = pred.iloc[0]
print('Valores de la postulación 1:')
print(fila[COLS].to_string())
print('\nProbabilidad que dio el árbol:', fila.prob_arbol, '| respuesta real:', 'aceptada' if fila.aceptada else 'rechazada')
print('\nAhora sigue el árbol a mano:')
print(open('2-modelo/resultados/arbol.txt', encoding='utf-8').read())

## Ejercicio 8. Con los datos de verdad

Vuelve a la sección 3, pon `USAR_BASE = True` y corre todo otra vez. Antes de leer el resultado,
apunta qué esperas. Luego compara:

| Si sale… | Quiere decir… |
|---|---|
| Esperando datos, «faltan N» | Todavía no hay 30 respondidas (8 y 8). El modelo no da consejos, como pide el PDF |
| Árbol ≈ flojo, ROC-AUC ≈ 0,5 | Las respuestas de la base tampoco tienen patrón (como en el ejercicio 2) |
| Árbol le gana al flojo, ROC-AUC > 0,6 | ¡Hay patrón! Mira qué pesa más y si tiene sentido (como en el ejercicio 1) |

## Lista para revisar contra el PDF

- [ ] La tabla solo tiene postulaciones aceptadas o rechazadas (sección 3).
- [ ] No hay nombre, correo, teléfono, foto, edad, sexo ni empresa en la tabla (`tabla.columns`).
- [ ] El árbol tiene máximo 4 niveles y cada hoja al menos 5 casos (`arbol.txt` y `fig_arbol.png`).
- [ ] Se compara con la logística y con el flojo, en 5 pliegues por vacante (sección 5).
- [ ] Si no le gana al flojo, `dar_consejos` es `false` y la página lo dice.
- [ ] En la página, «Pruébalo» nunca muestra un porcentaje, solo buenas, regulares o bajas.
- [ ] El navegador da lo mismo que Python (sección 6: 151 de 151).